<div style="
    color: red; 
    border: 2px solid black; 
    border-radius: 15px; 
    padding: 8px; 
    font-size: 24px; 
    font-weight: bold; 
    text-align: center; 
    margin: 20px 0;">
    <div style="
        border: 2px solid red; 
        border-radius: 10px; 
        padding: 10px;">
        <img src="https://www.polytecsousse.tn/wp-content/uploads/2024/12/poly-tec.jpg" alt="Datailab Logo" style="max-width: 100%; height: auto;">
        <h2 style="color: black; margin-top: 10px;">Exercice Pratique : Diagnostic et Correction</h2>
    </div>
</div>


In [9]:
import piplite
await piplite.install('scikit-learn')
print("Installation terminée!")

Installation terminée!


In [ ]:
# Fonction de diagnostic du compromis biais-variance
def diagnose_bias_variance(model, X_train, X_test, y_train, y_test, cv=5):
    """
    Diagnostique le compromis biais-variance d'un modèle
    
    Parameters:
    -----------
    model : estimator
        Modèle à évaluer
    X_train, y_train : array
        Données d'entraînement
    X_test, y_test : array
        Données de test
    cv : int
        Nombre de folds pour la validation croisée
    """
    
    # 1. Entraînement sur le jeu complet
    model.fit(X_train, y_train)
    
    # 2. Calcul des scores
    train_score = accuracy_score(y_train, model.predict(X_train))
    test_score = accuracy_score(y_test, model.predict(X_test))
    
    # 3. Validation croisée pour estimer la variance
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # 4. Diagnostic
    gap = train_score - test_score
    
    print(f"📊 Modèle: {model.__class__.__name__}")
    print(f"   Score train: {train_score:.3f}")
    print(f"   Score test:  {test_score:.3f}")
    print(f"   Écart:       {gap:.3f}")
    print(f"   CV score:    {cv_mean:.3f} (±{cv_std:.3f})")
    
    # 5. Interprétation
    if train_score < 0.7 and test_score < 0.7:
        print("   ⚠️  BIAIS ÉLEVÉ : Modèle trop simple, sous-ajustement")
        print("   → Solution : Augmenter la complexité du modèle")
    elif gap > 0.15:  # Écart important entre train et test
        print("   ⚠️  VARIANCE ÉLEVÉE : Modèle trop complexe, surajustement")
        print("   → Solution : Régularisation, simplification")
    elif 0.05 < gap <= 0.15:
        print("   ⚠️  VARIANCE MODÉRÉE : Risque de surajustement")
        print("   → Solution : Légère régularisation recommandée")
    else:
        print("   ✅ BON COMPROMIS : Modèle bien ajusté")
    
    # 6. Recommandations spécifiques
    if 'DecisionTree' in model.__class__.__name__:
        if hasattr(model, 'max_depth'):
            if model.max_depth is not None and model.max_depth > 10:
                print("   💡 Arbre très profond → risque de variance élevée")
            elif model.max_depth is not None and model.max_depth <= 3:
                print("   💡 Arbre peu profond → risque de biais élevé")
    
    return {
        'train_score': train_score,
        'test_score': test_score,
        'gap': gap,
        'cv_mean': cv_mean,
        'cv_std': cv_std
    }

In [ ]:
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np
# Données non-linéaires
X, y = make_moons(n_samples=1000, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# Test de différents modèles
models = {
    'Arbre Profond (Variance)': DecisionTreeClassifier(max_depth=20),
    'Arbre Simple (Biais)': DecisionTreeClassifier(max_depth=2),
    'Arbre Optimisé': DecisionTreeClassifier(max_depth=5, min_samples_split=10),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5),
    'SVM Linéaire (Biais)': SVC(kernel='linear', C=1),
    'SVM RBF (Variance)': SVC(kernel='rbf', C=10, gamma=10)
}

print("🧪 TEST DU COMPROMIS BIAS-VARIANCE")
print("=" * 50)

for name, model in models.items():
    diagnose_bias_variance(model, X_train, X_test, y_train, y_test)
    print("-" * 40)